# Anggota 3 — Pencarian passage

**Affan Abyarahman**

Sistem tanya jawab aturan akademik dan ujian FILKOM tanpa LLM. Notebook ini memeriksa kecocokan korpus Orie dan token Rio, membangun indeks, dan menguji pengurutan passage.

**Colab:** unggah satu file `PBA_FILKOM_Anggota1-3.zip` ke panel Files. Jalankan **Run all**.

**Lokal:** ekstrak paket gabungan dengan tiga folder anggota sejajar. Buka notebook dari folder proyek atau folder Anggota 3. Gunakan Python 3.10+ dan scikit-learn.

Sumber: Pedoman Akademik FILKOM 2020, Tata Tertib UTS/UAS 2016, dan Pedoman Penilaian PTIIK 2015.

In [1]:
from pathlib import Path
from zipfile import ZipFile
import os

folder_colab = Path("/content")
if folder_colab.is_dir():
    arsip = folder_colab / "PBA_FILKOM_Anggota1-3.zip"
    folder_proyek = folder_colab / "PBA_FILKOM_Anggota1-3"
    bersama = [
        "Anggota1_Korpus/data/passages.jsonl",
        "Anggota1_Korpus/data/dokumen.json",
        "Anggota2_Pertanyaan/preprocess.py",
        "Anggota2_Pertanyaan/question_processing.py",
        "Anggota2_Pertanyaan/data/pertanyaan_anggota2.json",
        "Anggota2_Pertanyaan/output/passages_preprocessed.jsonl",
        "Anggota3_Retrieval/retrieval.py",
        "Anggota3_Retrieval/data/pertanyaan_anggota3.json",
    ]
    if arsip.is_file():
        with ZipFile(arsip) as paket:
            for nama in bersama:
                paket.extract(nama, folder_proyek)
    if not all((folder_proyek / nama).is_file() for nama in bersama):
        raise FileNotFoundError(f"Unggah {arsip.name} di panel Files, lalu jalankan ulang.")
    os.chdir(folder_proyek)
    print("Data dan modul siap.")


In [2]:
from pathlib import Path
import json
import sys

cwd = Path.cwd().resolve()
root = next((
    folder for folder in (cwd, *cwd.parents)
    if (folder / "Anggota1_Korpus/data/passages.jsonl").is_file()
    and (folder / "Anggota2_Pertanyaan/preprocess.py").is_file()
), None)
if root is None:
    raise FileNotFoundError("Buka notebook dari folder proyek yang berisi folder Orie dan Rio.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from Anggota2_Pertanyaan.preprocess import preprocess_passage, VERSI_PREPROCESS
from Anggota2_Pertanyaan.question_processing import proses_pertanyaan

folder_korpus = root / "Anggota1_Korpus/data"
folder_rio = root / "Anggota2_Pertanyaan"

def baca_jsonl(path):
    with path.open(encoding="utf-8-sig") as file:
        return [json.loads(baris) for baris in file if baris.strip()]

passages = baca_jsonl(folder_korpus / "passages.jsonl")
dokumen = json.loads((folder_korpus / "dokumen.json").read_text(encoding="utf-8-sig"))
ekspor_rio = baca_jsonl(folder_rio / "output/passages_preprocessed.jsonl")
passage_by_id = {p["id"]: p for p in passages}
ekspor_by_id = {p["passage_id"]: p for p in ekspor_rio}

assert len(passage_by_id) == len(passages), "ID passage Orie tidak unik"
assert len(ekspor_by_id) == len(ekspor_rio), "ID ekspor Rio tidak unik"
assert passage_by_id.keys() == ekspor_by_id.keys(), "Daftar ID Orie dan Rio berbeda"
for p in passages:
    hasil = ekspor_by_id[p["id"]]
    assert p["dok_id"] in dokumen, p["id"]
    assert p["dok_id"] == hasil["dok_id"] and p["halaman"] == hasil["halaman"], p["id"]
    assert hasil["versi_preprocess"] == VERSI_PREPROCESS, p["id"]
    assert hasil["token_pencarian"] == preprocess_passage(p), p["id"]

print(f"{len(passages)} passage cocok: ID, token, versi preprocessing, dan metadata lokasi.")

440 passage cocok: ID, token, versi preprocessing, dan metadata lokasi.


## 1. Pahami masukan Rio dan teks asli Orie

Query untuk pencarian sudah tersedia sebagai `hasil["token_pencarian"]`. Gunakan daftar itu langsung. Untuk korpus, pakai `preprocess_passage(p)` yang mengolah gabungan judul subbab, konteks, dan isi. Simpan teks asli untuk Anggota 4 agar angka seperti `2,00`, pembanding, dan huruf mutu tetap bisa dibaca.

In [3]:
hasil = proses_pertanyaan("Berapa lama waktu maksimal untuk Ujian Skripsi?")
print("Status:", hasil["status"])
print("Token pencarian:", hasil["token_pencarian"])
print("Tipe jawaban:", hasil["tipe_jawaban"])
contoh = passage_by_id["pa2020-VI-D-10"]
print("Teks asli calon bukti:", contoh["teks"])
print("Sumber:", dokumen[contoh["dok_id"]]["judul"], "— halaman PDF", contoh["halaman"])

Status: SIAP_RETRIEVAL
Token pencarian: ['lama', 'waktu', 'maksimal', 'ujian', 'skripsi']
Tipe jawaban: DURASI
Teks asli calon bukti: 10. Waktu yang disediakan untuk Ujian Skripsi paling lama 75 menit untuk masing-masing mahasiswa.
Sumber: Buku Pedoman Akademik Fakultas Ilmu Komputer Universitas Brawijaya — halaman PDF 43


## 2. Coba TF-IDF dan cosine pada data kecil

TF-IDF memberi bobot lebih besar pada kata yang membantu membedakan passage. Cosine similarity melihat kemiripan antara vektor pertanyaan dan passage. Skor dipakai untuk mengurutkan kandidat, bukan peluang bahwa jawabannya benar.

Contoh ini sudah menggunakan daftar token, jadi tokenizer bawaan tidak perlu memecahnya lagi. Pertanyaan harus menggunakan kosakata indeks yang sama: panggil `transform`, bukan `fit_transform`.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def pakai_token_langsung(tokens):
    return tokens

contoh_tokens = [["ujian", "skripsi", "waktu"], ["cuti", "akademik"]]
contoh_vectorizer = TfidfVectorizer(analyzer=pakai_token_langsung)
contoh_matriks = contoh_vectorizer.fit_transform(contoh_tokens)
contoh_query = contoh_vectorizer.transform([["ujian", "skripsi"]])
print(cosine_similarity(contoh_query, contoh_matriks)[0].round(3))

[0.816 0.   ]


## 3. Buat indeks dan cari passage

Kode utama ada di `retrieval.py`. Indeks dibangun sekali dari judul subbab, konteks, dan teks yang diolah dengan fungsi Rio. Query memakai token pencarian dari Rio. Skor cosine dipakai untuk mengurutkan kandidat; skor 0 tidak ditampilkan.

Setiap kandidat membawa teks asli, metadata sumber, `passage_id`, `rank`, dan `score`. Anggota 4 memakai kandidat itu untuk mengambil jawaban.

In [5]:
from Anggota3_Retrieval.retrieval import Retrieval

retrieval = Retrieval(passages, dokumen)
cari_passage = retrieval.cari_passage
for p in cari_passage(hasil, top_k=3):
    print(f'{p["rank"]}. {p["passage_id"]} | skor {p["score"]:.3f}')
    print(p["teks"])
    print()

1. pa2020-VI-D-10 | skor 0.458
10. Waktu yang disediakan untuk Ujian Skripsi paling lama 75 menit untuk masing-masing mahasiswa.

2. pa2020-VI-D-9 | skor 0.355
9. Tugas dan Kewajiban Majelis Ujian Skripsi a. Ketua: i. Memimpin jalannya Ujian Skripsi. ii. Melakukan rekapitulasi penilaian akhir Ujian Skripsi. iii. Bertanggung jawab kepada Dekan atas pelaksanaan Ujian Skripsi dan menandatangani Berita Acara Ujian Skripsi. iv. Bertindak sebagai Penguji. b. Penguji: i. Melakukan penilaian Ujian Skripsi dengan mengacu pada Panduan Skripsi. c. Dosen Pembimbing: i. Mendampingi dan memberikan arahan yang diperlukan terhadap mahasiswa bimbingannya. ii. Melakukan penilaian Ujian Skripsi dengan mengacu pada Panduan Skripsi.

3. pa2020-VI-D-11 | skor 0.339
11. Penilaian Ujian Skripsi mengacu pada Panduan Skripsi.



## 4. Uji pemeringkatan dengan pertanyaan Rio

Bandingkan ID passage yang ditemukan dengan ID rujukan. Lima besar dipakai sebagai batas awal. Peringkat penuh ikut ditampilkan supaya terlihat apakah bukti ada di bawah batas itu.

In [6]:
uji_rio = json.loads(
    (folder_rio / "data/pertanyaan_anggota2.json").read_text(encoding="utf-8-sig")
)
jumlah_top5 = 0
for kasus in uji_rio:
    info = proses_pertanyaan(kasus["pertanyaan"])
    kandidat = cari_passage(info, top_k=len(passages))
    posisi = next((
        p["rank"] for p in kandidat if p["passage_id"] in kasus["passage_id"]
    ), None)
    if posisi is not None and posisi <= 5:
        jumlah_top5 += 1
        status = str(posisi)
    else:
        status = f"tidak masuk top-5 (peringkat: {posisi or 'tidak ditemukan'})"
    print(kasus["no"], kasus["pertanyaan"], "→", status)
print(f"Bukti masuk top-5: {jumlah_top5}/{len(uji_rio)}")

1 Berapa sks minimal untuk menyelesaikan studi sarjana? → 3
2 Berapa lama waktu maksimal untuk Ujian Skripsi? → 1
3 Apa itu skripsi? → tidak masuk top-5 (peringkat: 19)
4 Apa nilai huruf untuk nilai angka 78? → tidak masuk top-5 (peringkat: 6)
Bukti masuk top-5: 2/4


**Hasil percobaan awal dengan 440 passage, TF-IDF unigram, dan cosine:**

| Pertanyaan Rio | Peringkat bukti pertama: isi saja | Judul + konteks + isi |
|---|---:|---:|
| SKS minimal menyelesaikan studi | 5 | 3 |
| Durasi maksimal Ujian Skripsi | 1 | 1 |
| Definisi skripsi | 36 | 19 |
| Nilai huruf untuk 78 | 6 | 6 |

Gabungan judul dan konteks membawa dua bukti lebih dekat. Pada gabungan ini, bukti masuk lima besar untuk **2 dari 4** pertanyaan. Ini pemeriksaan awal dengan contoh Rio; jangan sebut sebagai akurasi jawaban atau hasil evaluasi 20 soal.

**Perbaikan yang bisa dicoba setelah baseline:**

- **Definisi:** untuk tipe `DEFINISI`, cari pola istilah diikuti “adalah” atau “merupakan” pada teks asli, lalu prioritaskan hasil tersebut. Kata “skripsi” sendiri muncul pada banyak aturan.
- **Konversi nilai:** beri jalur khusus pencarian bagian tabel konversi saat tipe `NILAI_HURUF`. Kirim seluruh kandidat baris dalam satu tabel atau cari lebih dari lima baris khusus tipe ini. Anggota 4 yang mengecek apakah angka masuk rentang. Jangan menebak huruf dari skor cosine.
- Uji lagi dengan variasi pertanyaan yang belum dipakai menyusun aturan sebelum menetapkan metode akhir.

## 5. Pertanyaan kontribusi Anggota 3

Tiga pertanyaan ini tersimpan di `data/pertanyaan_anggota3.json`. Jawabannya mengacu pada edisi yang dipakai korpus. Untuk cuti, tipe `WAKTU` dari Rio berarti tenggat relatif terhadap daftar ulang.

In [7]:
file_pertanyaan = root / "Anggota3_Retrieval/data/pertanyaan_anggota3.json"
uji_affan = json.loads(file_pertanyaan.read_text(encoding="utf-8-sig"))
for row in uji_affan:
    info = proses_pertanyaan(row["pertanyaan"])
    assert info["tipe_jawaban"] == row["tipe"]
    assert info["jenis_pertanyaan"] == row["jenis_pertanyaan"]
    p = passage_by_id[row["passage_id"][0]]
    assert row["bukti"] in p["teks"]
    print(row["pertanyaan"])
    print(row["jawaban"], "|", p["id"], "| halaman PDF", p["halaman"])
    print()

Berapa lama waktu maksimal untuk menyelesaikan studi sarjana?
14 semester sejak pertama kali terdaftar sebagai mahasiswa. | pa2020-II-I-8 | halaman PDF 21

Kapan batas pengajuan cuti akademik setelah daftar ulang?
Paling lambat 1 bulan setelah masa daftar ulang berakhir. | pa2020-III-G-2 | halaman PDF 26

Apa rumus indeks prestasi semester?
IP = Σ (K_i × Na_i) / Σ K_i. | pa2020-II-H-3 | halaman PDF 20



## 6. Poin yang perlu kamu jelaskan di laporan

- **Bab II:** peran retrieval, gagasan TF-IDF, dan cosine similarity.
- **Bab III:** alur dari token Rio sampai kandidat passage, termasuk alasan memasukkan judul dan konteks.
- **Bab IV:** cara membaca data Orie, fungsi Rio yang dipakai ulang, fungsi yang kamu tulis sendiri, struktur keluaran, dan penanganan hasil kosong.
- **Bahan Bab V:** peringkat bukti, contoh yang berhasil dan gagal, serta alasan variasi kata atau bentuk tabel dapat menurunkan peringkat. Evaluasi akhir dilakukan dengan data uji yang disepakati kelompok.